# Летний лагерь "Заря-2"
Интерактивная текстовая новелла на чистом Python (ООП, ветвления, циклы, структуры данных).

In [ ]:
class Character:
    def __init__(self, name, role):
        self.name = name
        self.role = role


class PlayerState:
    def __init__(self, player_name):
        self.player_name = player_name
        self.courage = 0
        self.trust = 0
        self.inventory = []
        self.clues = set()
        self.route = []
        self.ending = ''


class NovelGame:
    def __init__(self):
        self.characters = [
            Character('Артём', 'вожатый'),
            Character('Лера', 'активистка'),
            Character('Игорь', 'техник')
        ]
        self.locations = {
            'медпункт': 'Там тихо и пахнет лекарствами.',
            'сцена': 'За кулисами лежат старые декорации.',
            'радиорубка': 'Пыльный пульт и дневник дежурств.',
            'лодочная': 'Под настилом спрятан металлический ящик.'
        }
        self.state = None

    def ask_choice(self, prompt, options):
        print(prompt)
        for key, value in options.items():
            print(str(key) + '. ' + value)
        answer = input('> ').strip()
        while answer not in options:
            print('Нужно выбрать один из предложенных вариантов.')
            answer = input('> ').strip()
        return answer

    def intro(self):
        name = input('Введите имя главного героя: ').strip()
        if name == '':
            name = 'Семён'
        self.state = PlayerState(name)
        print('')
        print("Автобус останавливается у ворот лагеря 'Заря-2'.")
        print('Тебя встречает вожатый Артём и предлагает выбрать, чем заняться в первый вечер.')

    def first_choice(self):
        answer = self.ask_choice(
            'Первое решение:',
            {'1': 'Пойти помогать на сцену', '2': 'Уйти к озеру в одиночку'}
        )
        if answer == '1':
            self.state.trust += 2
            self.state.courage += 1
            self.state.inventory.append('фонарик')
            self.state.route.append('помощь на сцене')
            print('Артём благодарит тебя и даёт фонарик.')
        else:
            self.state.courage += 2
            self.state.trust -= 1
            self.state.inventory.append('карта лагеря')
            self.state.route.append('прогулка к озеру')
            print('На берегу ты находишь старую карту лагеря.')

    def investigate(self):
        print('')
        print('Ночью в лагере пропадает ключ от архива. Нужно проверить 2 места.')
        checked = 0
        for place in self.locations:
            if checked == 2:
                break
            answer = self.ask_choice(
                "Проверить локацию '{}'?".format(place),
                {'1': 'Да', '2': 'Пропустить'}
            )
            if answer == '1':
                checked += 1
                self.state.route.append('осмотр: ' + place)
                self.state.clues.add(place)
                self.state.inventory.append('улика из ' + place)
                print(self.locations[place])

    def decode_radio(self):
        print('')
        print('В радиорубке мигает старый передатчик. Чтобы открыть архив, нужен код из 4 цифр.')
        attempts = 3
        while attempts > 0:
            code = input('Введите код: ').strip()
            if code == '1967':
                self.state.trust += 2
                self.state.clues.add('код-1967')
                self.state.route.append('передатчик разблокирован')
                print('Код верный. Архив открыт.')
                return True
            attempts -= 1
            print('Неверно. Осталось попыток: ' + str(attempts))
        self.state.route.append('передатчик не разблокирован')
        return False

    def final_branch(self, radio_success):
        if 'фонарик' in self.state.inventory and 'лодочная' in self.state.clues:
            self.state.courage += 1
        if 'карта лагеря' in self.state.inventory and 'сцена' in self.state.clues:
            self.state.trust += 1

        if radio_success and self.state.trust >= 3 and self.state.courage >= 2:
            self.state.ending = 'Хорошая концовка: ты раскрываешь тайну архива и становишься легендой лагеря.'
        elif radio_success and self.state.courage >= 2:
            self.state.ending = 'Нейтральная концовка: ключ найден, но часть правды так и остаётся в тени.'
        else:
            self.state.ending = 'Плохая концовка: архив закрывают, а тайна лагеря уходит в небытие.'

    def cleanup_inventory(self):
        if len(self.state.inventory) > 5:
            self.state.inventory.pop(0)
        if 'карта лагеря' in self.state.inventory and 'фонарик' in self.state.inventory:
            self.state.inventory.remove('карта лагеря')
            self.state.inventory.append('маршрут с пометками')

    def build_report(self):
        report = {}
        report['герой'] = self.state.player_name
        report['смелость'] = self.state.courage
        report['доверие'] = self.state.trust
        report['улики'] = list(self.state.clues)
        report['предметы'] = self.state.inventory
        report['маршрут'] = self.state.route
        report['концовка'] = self.state.ending
        return report

    def save_result(self, report):
        text = []
        text.append('Герой: ' + report['герой'])
        text.append('Смелость: ' + str(report['смелость']))
        text.append('Доверие: ' + str(report['доверие']))
        text.append('Маршрут:')
        for step in report['маршрут']:
            text.append('- ' + step)
        text.append('Предметы: ' + ', '.join(report['предметы']))
        text.append('Улики: ' + ', '.join(report['улики']))
        text.append('Финал: ' + report['концовка'])

        with open('novel_result.txt', 'w', encoding='utf-8') as file:
            file.write('\n'.join(text))

    def run(self):
        self.intro()
        self.first_choice()
        self.investigate()
        radio_success = self.decode_radio()
        self.cleanup_inventory()
        self.final_branch(radio_success)
        report = self.build_report()
        self.save_result(report)
        print('')
        print(self.state.ending)
        print('Итог автоматически сохранён в файл novel_result.txt')


game = NovelGame()
game.run()
